# Turing Engine on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/intutic/turing/blob/master/demo/turing_quickstart_colab.ipynb)

[Turing Engine](https://github.com/intutic/turing) is an LLM inference server for KV-cache and quantization research. It can store the KV cache in 8 or 4 bits, finds the layers a low-bit cache would break and keeps those in full precision, and is tested against Hugging Face's implementation of each model it serves.

This notebook installs the released package and, on this runtime:

1. starts `turing serve` and talks to it through the OpenAI, Anthropic and Ollama APIs;
2. shows which layers an int8 and an int4 KV cache keep in full precision, and how many tokens each holds;
3. asks the same questions with the int4 cache;
4. compares Turing's logits with Hugging Face's.

Every number it prints is measured here when you run it. Use a GPU runtime (**Runtime → Change runtime type → T4 GPU**); on a CPU it runs, slowly, with a smaller model.

In [ ]:
!pip install -q turing-engine openai

import gc, subprocess, sys, time
import requests, torch, turing

print("turing", turing.__version__, "| torch", torch.__version__, "| python", sys.version.split()[0])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none, running on the CPU")
MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
# A T4 (compute capability 7.5) has no fast bf16, so the model runs in fp16 there.
DTYPE = "fp16" if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] < 8 else None
print("model:", MODEL, "| dtype:", DTYPE or "the checkpoint's")

## 1. Start the server

`turing serve` downloads the model, measures which KV-cache layers to keep in full precision, compiles and warms up its kernels, and then listens. On a GPU the KV cache is int8 unless you choose otherwise.

In [ ]:
BASE = "http://127.0.0.1:8000"

def start(*args):
    cmd = ["turing", "serve", "--model", MODEL, "--host", "127.0.0.1", "--port", "8000", *args]
    if DTYPE:
        cmd += ["--dtype", DTYPE]
    log = open("turing_serve.log", "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while True:
        try:
            if requests.get(BASE + "/health", timeout=2).ok:
                break
        except requests.RequestException:
            pass
        if proc.poll() is not None:
            print(open("turing_serve.log").read()[-3000:])
            raise RuntimeError("turing serve exited")
        time.sleep(2)
    print(f"ready after {time.time() - t0:.0f} s: {' '.join(cmd[2:])}")
    return proc

def stop(proc):
    proc.terminate()
    proc.wait(timeout=120)

server = start()
print(requests.get(BASE + "/health").json())

## 2. OpenAI, Anthropic and Ollama APIs on one port

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE + "/v1", api_key="unused")  # no key is checked unless the server has --api-key
QUESTIONS = [
    "What is the capital of Australia? Answer in one word.",
    "Write a Python one-liner that reverses a string s.",
    "Which planet is the largest in the Solar System? Answer in one word.",
]

def ask(question, max_tokens=160):
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": question}],
                                       max_tokens=max_tokens, temperature=0)
    return r.choices[0].message.content.strip()

int8_answers = [ask(q) for q in QUESTIONS]
for q, a in zip(QUESTIONS, int8_answers):
    print(f"Q: {q}\nA: {a}\n")

# One streamed request: time to the first token and the decode rate, measured here.
t0, first, n = time.perf_counter(), None, 0
stream = client.chat.completions.create(
    model=MODEL, messages=[{"role": "user", "content": "Explain what a KV cache is in three sentences."}],
    max_tokens=128, temperature=0, stream=True, stream_options={"include_usage": True})
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content and first is None:
        first = time.perf_counter()
    if chunk.usage:
        n = chunk.usage.completion_tokens
t1 = time.perf_counter()
print(f"first token after {1000 * (first - t0):.0f} ms, then {n - 1} tokens at {(n - 1) / (t1 - first):.1f} tok/s")

In [ ]:
r = requests.post(BASE + "/v1/messages", headers={"anthropic-version": "2023-06-01"},
                  json={"model": MODEL, "max_tokens": 32, "messages": [{"role": "user", "content": "Say hello in French."}]})
print("Anthropic /v1/messages:", r.json()["content"][0]["text"])

r = requests.post(BASE + "/api/chat", json={"model": MODEL, "stream": False, "options": {"num_predict": 32, "temperature": 0},
                                             "messages": [{"role": "user", "content": "Say hello in Spanish."}]})
print("Ollama /api/chat:", r.json()["message"]["content"])

## 3. Which layers a low-bit KV cache would break

When the KV cache is stored in fewer bits, Turing round-trips each layer's keys and values through that format on its own and measures how much the model's next-token distribution changes (the KL divergence, on random tokens). Layers above 0.02 stay in the model's dtype. Here the server stops and the same model is loaded in this process to look at that choice. Each pool takes the same share of free GPU memory, so the token counts compare the formats at equal memory.

In [ ]:
stop(server)
from turing.models.hf_loader import RealHuggingFaceLoader

model, tokenizer = RealHuggingFaceLoader.load_hf_model_into_turing(MODEL, device=DEVICE, dtype=DTYPE)
dtype_name = str(model.embed_tokens.weight.dtype).split(".")[-1]
print(f"{MODEL}: {model.config.num_layers} layers in {dtype_name}\n")
for kv in ("native", "int8", "int4"):
    model.config.kv_cache_dtype, model.config.kv_native_layers = kv, None
    st = model.kv_pool().get_stats()
    tokens = st["total_pages"] * st["page_size"]
    kept = f"all {model.config.num_layers}" if kv == "native" else (st["native_layers"] or "none")
    print(f"{kv:>6}: {tokens:>9,} tokens, {st['bytes_per_page'] / st['page_size'] / 1024:5.1f} KiB per token, "
          f"layers in {dtype_name}: {kept}")
    kl = getattr(model, "_kv_layer_kl", None) if kv != "native" else None
    if kl:
        worst = sorted(range(len(kl)), key=lambda i: -kl[i])[:3]
        print("        most damaged by this format alone (KL):", ", ".join(f"layer {i} {kl[i]:.3f}" for i in worst))

## 4. The same questions with the int4 cache

In [ ]:
del model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

server = start("--kv-cache-dtype", "int4")
int4_answers = [ask(q) for q in QUESTIONS]
for q, a8, a4 in zip(QUESTIONS, int8_answers, int4_answers):
    print(f"Q: {q}\n  int8: {a8}\n  int4: {a4}\n")
stop(server)

## 5. Turing against Hugging Face

With the KV cache in the model's dtype, Turing computes the same function as Hugging Face's implementation. In half precision the two differ by rounding, so this checks that the logits are close and that both predict the same next token at every position of a prompt.

In [ ]:
from transformers import AutoModelForCausalLM

model, _ = RealHuggingFaceLoader.load_hf_model_into_turing(MODEL, device=DEVICE, dtype=DTYPE)
dtype = model.embed_tokens.weight.dtype
hf = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(DEVICE).eval()
text = "A KV cache stores the keys and values of the tokens a model has already read, so each new token"
ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
with torch.inference_mode():
    ours, ref = model(ids)[0].float(), hf(ids).logits.float()
agree = (ours.argmax(-1) == ref.argmax(-1)).float().mean().item()
print(f"{ids.shape[1]} positions in {str(dtype).split('.')[-1]}: largest logit difference {(ours - ref).abs().max().item():.3f}, "
      f"same next token at {agree:.0%} of positions")

## Next

- Documentation: [intutic.github.io/turing](https://intutic.github.io/turing/), including the [comparison with vLLM and llama.cpp](https://intutic.github.io/turing/comparison/) and the [serving reference](https://intutic.github.io/turing/serving/).
- On your own machine: `pip install turing-engine`, or the Docker images `ghcr.io/intutic/turing:v2.1.0` (CPU) and `ghcr.io/intutic/turing:v2.1.0-cuda`.
- Turing Engine is under the Business Source License 1.1: free for non-production use such as research, development and benchmarking ([licensing](https://intutic.github.io/turing/licensing/)).